   ... normal to ... Computer architecture is notoriously dense, with a lot of
   moving parts between the software we write and the physical silicon that 
   executes it.

   ... high-level refresher of the first two lectures to help you get your 
   bearings again.


---   
LECTURE 1: ARCHITECTURE AND INSTRUCTIONS
   This lecture sets the stage for the entire module. It answers the fundamental
   question of why we care about hardware when we usually just write code, and
   introduces the bridge between the two.

   - THE BIG PICTURE: The core concept here is the INSTRUCTION SET ARCHITECTURE
     (ISA). The ISA is the critical interface or "contract" between software
     and hardware. It defines the set of instructions the processor can
     understand. 
   - WHY ARCHITECTURE MATTERS: The lecture motivates this by looking at modern
     accelerators. Standard CPU's aren't always enough anymore. The timestamps
     mention TPUs (Tensor Processing Units), which are highly specialised chips
     designed explicitly to accelerate matrix multiplications for machine
     learning models (like the Transformers powering GPT). Cloud providers
     (Microsoft, Amazon) rely heavily on these custom architectures.
   - THE RISC-V ARCHITECTURE: This module focuses on RISC-V (Reduced Instruction
     Set Computer). It is an open-source ISA that prioritises simplicity and
     efficiency.
      - R-TYPE INSTRUCTIONS: These are "Register-type" instructions used for
        standard arithmetic and logical operations (like `add`, `sub`, `and`).
        They typically take two source registers, perform an operation, and
        store the result in a destination register. 
      - FROM CODE TO SILICON: You looked at how high-level control flow (like an
        `if`-statement in C or C++) gets compiled down itno a sequence of these 
        basic RISC-V instructions (using branches and jumps).
      - THE DATAPATH: This is the physical wiring, ALUs, and memory components
        that actually execute the instructions. 





---
ISA: Instruction Set Architecture
   The ISA is the INTERFACE between software and hardware -- like an API for the
   processor. It defines:

   * What INSTRUCTIONS the processor understands (add, subtract, load, store,
     jump...)
   * What REGISTERS are available (small, fast storage inside the processor)
   * How MEMORY is addressed
   * The ENCODING FORMAT of instructions (how they're represented in binary)

   The key insight: DIFFERENT HARDWARE CAN IMPLEMENT THE SAME ISA. Your laptop
   and a server can both run the same RISC-V code, even if their internal
   circuits are completely different. This is what makes software portable.


DESIGN APPROACHES: RISC vs. CISC
   There are two major philosophies:


CISC (Complex Instruction Set Computer) -- e.g., x86 (Intel/AMD). Instructions
   can be very complex ("Multiply these two memory values and store the result
   back to memory in one instruction"). Instructions vary in length and can take
   many cycles.


RISC (Reduces Instruction Set Computer) -- e.g., RISC-V, ARM. Instructions are
   simple, fixed-length, and typically execute in one cycle. If you need 
   something complex, you combine multiple simple instructions. This makes the
   hardware much simpler and easier to pipeline (which you'll cover later).


   Your module uses RISC-V as the teaching architecture because it's clean, 
   open-source, and designed specifically to be easy to understand. 



---
RISC-V Architecture Basics
   RISC-V has 32 REGISTERS (x0 through x31), each 32 or 64 bits wide. Register
   x0 is hardwired to ALWAYS BE ZERO -- this is surprisingly useful (you'll
   see why). Instructions operate on registers, not directly on memory, 
   following the LOAD-STORE principle: you load data from memory into registers,
   do your computation, then store results back.


RISC-V R-type Instructions
   R-type instructions operate on THREE REGISTERS: two sources and one 
   destination. The format is:
```
add x5, x6, x7          # x5 = x6 + x7
sub x5, x6, x7          # x5 = x6 - x7
```

   The binary encoding has fixed fields:
```
|  funct7 (7)  |  rs2 (5)  |  rs1  (5)  |  funct3 (3)  |  rd (5)  | opcode (7)  |
```
   Every field has a fixed position -- this is what makes RISC decoding simple.
   The hardware doesn't have to "figure out" how long the instruction is or
   where the fields are. 

---

   ... they just look like alphabet soup until you break down what the 
   processor is actually doing with them. 

   ...

   breakdown of the R-Type instruction fields, reading from right to left (which
   is how the processor often starts looking at it):



---
- `opcode` (7 bits): This is the "Operation Code". It is the main category label
  for the instruction. When the processor reads these 7 bits, it says, "Ah, this
  is a standard Register-toRegister arithmetic operation" or "This is a memory
  load." For all R-Type arithmetic instructions, the opcode is exactly the same.

- `rd` (5 bits): This stands for REGISTER DESTINATION. This tells the processor
  where to save the final result of the calculation. It is 32 bits long because
  RISC-V has 32-general purpose registers, and it takes exactly 5 binary bits to
  count from 0 to 31.

- `funct3` (3 bits): This is a 3-bit "Function" code. Because the `opcode` only
  tells the processor the general category (e.g., "Arithmetic"), the `funct3`
  narrows it down further. For example, it might specify "This is an 
  addition/subtraction type of arithmetic" versus "This is a bitwise shift".

- `rs1` (5 bits): This stands for REGISTER SOURCE 1. It tells the processor
  which register holds the first input value for the math operation.

- `rs2` (5 bits): This stands for REGISTER SOURCE 2. It tells the processor 
  which register holds the second input value.

- `funct7` (7 bits): This is a 7-bit "Function" code, and it acts as the final
  tie-breaker. For example, the `add` and `sub` (subtract) instructions share
  the exact same `opcode` and the exact same `funct3`. The only way the
  processor knows whether to add or subtract is by looking at `funct7`.


WHY SPLIT IT UP LIKE THIS?
   In complex architectures like x86 (Intel/AMD), an instruction can be anywhere
   from 1 to 15 bytes long, and the fields move around. The hardware ahs to do a
   lot of complicated, slow work just to figure out what the instructions say.

   With RISC-V, the fields are in FIXED POSITIONS. When the 32-bit instructions
   hit the processor, the hardware wires can route bits 0-6 directly to the main
   control unit, and bits 7-11 directly to the register file's writee address, 
   all at the exact same time. It makes the physical silicon incredibly fast
   and simple to build. 


```
|  funct7 (7)  |  rs2 (5)  |  rs1  (5)  |  funct3 (3)  |  rd (5)  | opcode (7)  |
```

EXAMPLE: COMPILING IF-STATEMENTS
   This shows how high-level code maps to assembly. For example:

```
if (a == b) goto L1;
```

becomes:
```
beq x5, x6, L1          # branch to L1 if x5 == x6
```

   The key idea: EVERY high-level construct (if, while, for, function calls)
   eventually becomes a sequence of these simple instructions. The compiler's
   job is to do this translation efficiently.



---
LECTURE 2: PERFORMANCE    


...

#### 1. R-TYPE (Register)

WHAT IT IS
   R-type instruction takes TWO SOURCE REGISTERS, perform an operation, and 
   write the result to a DESTNATION REGISTER. No constants, no memory -- purely
   register-to-register.

WHEN TO USE R-TYPE
   When you're doing COMPUTATION BETWEEN TWO REGISTERS: `add`, `sub`, `and`, 
   `or`, `xor`, `sll` (shift left), `srl` (shift right), `slt` (set less than).
   These are the workhorses of any program.

WHEN NOT R-TYPE
   If one of your operand is a CONSTANT (like adding 5 to a register), you can't
   use R-type -- you need I-type. If you're accessing MEMORY, you need I-type
   (load) or S-type (store).



---
#### 2. I-TYPE (Immediate)

WHAT IT IS
   I-type insturctions use ONE SOURCE REGISTER and ONE CONSTANT VALUE EMBEDDED
   IN THE INSTRUCTION ITSELF (called an "immediate"). The result goes to a 
   destination register. 

   The encoding:
```
|  immediate (12)  |  rs1 (5)  |  funct3 (3)  |  rd (5)  |  opcode (7)  |
```
   Notice compared to R-type, the funct7 and rs2 fields are REPLACED by a single
   12-bit immediate. But rs1, funct3, and rd are in the EXACT SAME BIT POSITIONS
   as R-type -- this is deliberate, because it means the hardware that reads
   register numbers can be shared between instruction types.

   
TWO USE CASES SHOWN IN YOUR SLIDES
   USE CASE 1: Load from memory

```
ld x14, 8(x2)       # x14 = Memory[x2 + 8]
```
   
   Encoding:
```
| 000000001000 | 00010 | 010 | 01110 | 0000011 |
    imm = 8       x2     LW     x14    LOAD opcode
```

   How this works: the processor computes the EFFECTIVE ADDRESS by adding the
   immediate (8) to the value in rs1 (x2). Then it reads from that memory 
   address and stores the result in rd (x14).

   WHY THE OFFSET? In practice, x2 often points to the start of a data structure
   (like an array or a stack frame), and the immediate is the offset within that
   structure. For example, if x2 points to the base of a struct, `ld x14, 8(x2)`
   loads the field at byte offset 8.


USE CASE: Arithmetic with a constant
```
addi x15, x1, -50           # x15 = x1 - 50
```
         

Encoding:
```
 | 111111001110 | 00001 | 000 | 01111 | 0010011 |
    immm = -50     x1     ADD    x15    I-type opcode
```
   Notice the immediate is `111111001110` -- that's -5 in 12-BIT TWO'S 
   COMPLEMENT. The 12-bit value gets SIGN-EXTENDED to 32 bits before the
   addition, so negative numbers work correctly.

THE 12-BIT IMMEDIATE RANGE 
   With 12 bits (signed), you can encode constants from -2048 to +2047. This is
   enough for most offsets and small constants, but for larger values you'll 
   need U-tpe (see below).

WHEN TO USE I-TYPE
   Three situations: 
      * loading from memory (`ld`, `lw`, `lb`)
      * arithmetic with a small constant (`addi`, `andi`, `ori`, `slti`)
      * `jalr` -- jump to an address in a register plus offset

WHEN NOT TO USE I-TYPE
   If both operands are registers -> use R-type. If you need to STORE to memory
   --> use S-type (because stores need two register fields: the data and the
   base address, leaving no room for `rd`).

A SUBTLE BUT IMPORTANT POINT
   There's no `subi` instruction in RISC-V! If you want to subtract a constant,
   you just use `addi` with a NEGATIVE immediate: `addi x5, x6, -10` computes
   `x5 = x6 - 10`. This keeps the hardware simpler -- one fewer instruction to
   decode.



---
3. S-TYPE (Store)

WHAT IT IS
   S-type stores a register's value into MEMORY. It needs two register fields
   (the data to store AND the base address) plus an offset, but it DOESN'T
   WRITE TO ANY REGISTER -- so there's no `rd` field. 

THE ENCODING:
```
| imm[11:5] (7) | rs2 (5) | rs1 (5) | funct3 (3) | imm[4:0] (5) | opcode (7) |
```
   This is where it gets clever. The immediate is SPLIT INTO TWO PIECES: the 
   upper 7 bits go where `funct7` normally lives, and the lower 5 bits go where
   `rd` normally lives. Why? So that `rs1` and `rs2` remain in the EXACT SAME BIT
   POSITIONS as R-type and I-type. The hardware only needs one circuit to 
   extract register numbers, regardless of instruction type.

EXAMPLE FROM YOUR SLIDE:
```
sd x14, 8(x2)       # Memory[x2 + 8] = x14
```

Encoding:
```
| 0000000 | 01110 | 00010 | 010 | 01000 | 0100011 |
  imm[11:5]  x14     x2    SW   imm[4:0] STORE opcode
```
   The immediate 8 = `00000 01000`, split as: upper 7 bits = `0000000`, lower
   5 bits = `01000`. The processor reassembles these to get the offset 8, adds
   it to x2, and writes x14's value to that address.

WHEN TO USE S-TYPE
   Whenever you need to WRITE DATA TO MEMORY: `sd` (store doubleword/64-bit), 
   `sw` (store word/32-bit), `sh` (store halfword/16-bit), `sb` 
   (store byte/8-bit)   

```
ld  x14, 8(x2)   # I-type: Memory → Register  (x14 is the destination, rd)
sd  x14, 8(x2)   # S-type: Register → Memory  (x14 is the source, rs2)
```
   They look similar in assembly but COMPLETELY DIFFERENT FORMATS. Load has a 
   destination register (`rd`), so it fits I-type. Store has no destination
   register (it writes to memory, not a register), so it uses S-type with the
   split immediate. 
   


---
4. SB-TYPE / B-TYPE (Branch)
   
WHAT IT IS
   A variant of S-type used for CONDITIONAL BRANCHES (if-then jumps). It 
   compares two registers and jumps to a target address if the condition is
   met. 

EXAMPLE FROM YOUR SLIDE:
```
beq x19, x10, Label         # if x10 == x19, jump to Label
```
   The target address is encoded as an offset: PC + IMMEDIATE. In the slide, 
   Label is 16 bytes away, so the immediate encodes 16. 

THE ENCODING:
```
| imm[12] | imm[10:5] | rs2 | rs1 | funct3 | imm[4:1] | imm[11] | opcode |
```
   The immediate bits are SCRAMBLED across the instructions. This looks insa˜e,
   but there's a hardware reason: it keeps the most commonly needed bits in 
   positions that align with other instruction types, simplifying the circuit 
   that extracts them. 

   CRITICAL DETAIL: the immediate represents a BYTE OFFSET and BIT[0] is ALWAYS
   0 (because RISC-V instructions are always aligned to 2-byte boundaries). So
   the effective range is +- 4096 bytes from current PC -- roughly +- 1000 
   instructions.


AVAILABLE BRANCH CONDITIONS
   Using different funct3 values:

   * `beq` -- branch is equal
   * `bne` -- branch if not equal
   * `blt` -- branch if less than (signed)
   * `bge` -- branch if greater or equal
   * `bltu` -- branch if less than (unsigned)
   * `bgeu` -- branch if greater or equal (unsigned)


WHEN TO USE B-TYPE
   For implementing CONTROL FLOW, if-else, while loops, for loops. Every
   comparison-and-jump in your program becomes a B-type instruction. 










   







---


   In the context of this module, is is exactly 32 BITS in each of the 32 bits
   registers. 

   This specific baseline architecture you are studying is called RV32I 
   (Risc-V 32-bit Integer).

   ... breakdown of how that looks hardware-wise:
      * THE NUMBER OF REGISTERS: You have exactly 32 general-purpose registers,
        named x0 through x31.
      * THE SIZE OF EACH REGISTER: Every single one of those registers holds
        exactly 32 BITS (which is 4 bytes) of data.

   In RV32,  a 32-bit chunk of data is officially called a "WORD". This is a 
   super important term to remember because it explains why everything else in
   the datapath lines up so perfectly. Your registers are 32 bits wide, your
   ALU processes 32 bits at a time, and your instructions are exactly 32 bits
   long. Everything is designed around that magic number to keep the hardware
   simple and fast!       



--- ---
   Registers. There are 32 registers. RISC-V names them x0 through x3. We're 
   using the 64-bit version of the RISC-V ISA, so each register holds a 64-bit
   value. 